In [1]:
#AQI

!pip install pandas
!pip install polars
!pip install dask
!pip install pyarrow
!pip install duckdb

  Using cached pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached numpy-2.4.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
  Using cached polars-1.40.1-py3-none-any.whl.metadata (10 kB)
  Using cached polars_runtime_32-1.40.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached polars-1.40.1-py3-none-any.whl (828 kB)
Using cached polars_runtime_32-1.40.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (56.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]
  Using cached dask-2026.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2

In [2]:
import dask.dataframe as dd
import dask.dataframe as dd
import pyarrow as pa
import polars as ps
import numpy as np
import pandas as pd
import glob
import os
import duckdb

In [3]:
AQI_FILE = '/projects/illinois/ovcri/ncsa/cs2018/idph-ch-datasets/climate/interpolated_aqi_il_2017-24/aqi_all_years.parquet'

In [4]:
# Connect to DuckDB (runs in memory, no setup needed)
con = duckdb.connect()

In [5]:
print("=== COLUMN NAMES & TYPES ===")
print(con.execute(f"DESCRIBE SELECT * FROM '{AQI_FILE}'").df())

print("\n=== TOTAL ROWS ===")
count = con.execute(f"SELECT COUNT(*) FROM '{AQI_FILE}'").fetchone()[0]
print(f"  {count:,} rows")

print("\n=== FIRST 5 ROWS ===")
print(con.execute(f"SELECT * FROM '{AQI_FILE}' LIMIT 5").df())

=== COLUMN NAMES & TYPES ===
        column_name column_type null   key default extra
0      climate_date        DATE  YES  None    None  None
1            region     VARCHAR  YES  None    None  None
2              fips      BIGINT  YES  None    None  None
3               AQI      BIGINT  YES  None    None  None
4  Was_Interpolated     BOOLEAN  YES  None    None  None

=== TOTAL ROWS ===
  335,274 rows

=== FIRST 5 ROWS ===
  climate_date     region   fips  AQI  Was_Interpolated
0   2017-01-01      adams  17001   57              True
1   2017-01-01  alexander  17003   58              True
2   2017-01-01       bond  17005   62              True
3   2017-01-01      boone  17007   53              True
4   2017-01-01      brown  17009   57              True


In [6]:
print("=== NULL CHECK ===")
nulls = con.execute(f"""
SELECT 
    COUNT(*) as total_rows,
    {', '.join([f"COUNT(*) - COUNT({col}) AS {col}_nulls" for col in con.execute(f"DESCRIBE SELECT * FROM '{AQI_FILE}'").df()['column_name']])}
FROM '{AQI_FILE}'
""").df()

print(nulls)

=== NULL CHECK ===
   total_rows  climate_date_nulls  region_nulls  fips_nulls  AQI_nulls  \
0      335274                   0             0           0       6222   

   Was_Interpolated_nulls  
0                       0  


In [7]:
print("=== NUMERIC SUMMARY ===")
print(con.execute(f"""
SELECT * FROM '{AQI_FILE}'
""").df().describe())

=== NUMERIC SUMMARY ===
              climate_date           fips        AQI
count               335274  335274.000000   329052.0
mean   2021-07-02 00:00:00   17102.000000  45.360682
min    2017-01-01 00:00:00   17001.000000        0.0
25%    2019-04-02 00:00:00   17051.000000       37.0
50%    2021-07-02 00:00:00   17102.000000       44.0
75%    2023-10-02 00:00:00   17153.000000       52.0
max    2025-12-31 00:00:00   17203.000000      289.0
std                    NaN      58.886985  14.359091


In [8]:
cols = con.execute(f"DESCRIBE SELECT * FROM '{AQI_FILE}'").df()['column_name']

for col in cols:
    print(f"\n=== {col} UNIQUE VALUES ===")
    print(con.execute(f"""
    SELECT {col}, COUNT(*) as count
    FROM '{AQI_FILE}'
    GROUP BY {col}
    ORDER BY count DESC
    LIMIT 5
    """).df())


=== climate_date UNIQUE VALUES ===
  climate_date  count
0   2017-01-01    102
1   2017-01-02    102
2   2017-01-03    102
3   2017-01-04    102
4   2017-01-05    102

=== region UNIQUE VALUES ===
      region  count
0      perry   3287
1   woodford   3287
2  alexander   3287
3    jackson   3287
4  jefferson   3287

=== fips UNIQUE VALUES ===
    fips  count
0  17001   3287
1  17003   3287
2  17005   3287
3  17007   3287
4  17009   3287

=== AQI UNIQUE VALUES ===
   AQI  count
0   44  12867
1   42  11716
2   43  11647
3   41  11524
4   46  10836

=== Was_Interpolated UNIQUE VALUES ===
   Was_Interpolated   count
0              True  266409
1             False   68865


In [9]:
desc = con.execute(f"""
DESCRIBE SELECT * FROM '{AQI_FILE}'
""").df()

cols = desc['column_name'].tolist()

for col in cols:
    print(f"\n=== Checking {col} ===")
    
    print(con.execute(f"""
    SELECT {col}, COUNT(*) as count
    FROM '{AQI_FILE}'
    WHERE {col} IS NULL
    GROUP BY {col}
    LIMIT 5
    """).df())


=== Checking climate_date ===
Empty DataFrame
Columns: [climate_date, count]
Index: []

=== Checking region ===
Empty DataFrame
Columns: [region, count]
Index: []

=== Checking fips ===
Empty DataFrame
Columns: [fips, count]
Index: []

=== Checking AQI ===
    AQI  count
0  <NA>   6222

=== Checking Was_Interpolated ===
Empty DataFrame
Columns: [Was_Interpolated, count]
Index: []


In [10]:
champaign_df = con.execute(f"""
SELECT *
FROM '{AQI_FILE}'
WHERE LOWER(Region) = 'champaign'
""").df()

print(champaign_df.head())
print(f"Total rows: {len(champaign_df)}")

  climate_date     region   fips  AQI  Was_Interpolated
0   2017-01-01  champaign  17019   61             False
1   2017-01-02  champaign  17019   65             False
2   2017-01-03  champaign  17019   31             False
3   2017-01-04  champaign  17019   31             False
4   2017-01-05  champaign  17019   29             False
Total rows: 3287


In [11]:
print(con.execute(f"""
SELECT region, COUNT(*) 
FROM '{AQI_FILE}'
WHERE LOWER(REPLACE(TRIM(region), ' ', '')) IN (
    'champaign','clark','coles','cumberland','dewitt','douglas','edgar',
    'ford','iroquois','livingston','piatt','macon','mclean',
    'moultrie','shelby','vermilion'
)
GROUP BY region
ORDER BY COUNT(*) DESC
""").df())

        region  count_star()
0     moultrie          3287
1   livingston          3287
2        edgar          3287
3      douglas          3287
4        macon          3287
5    champaign          3287
6       shelby          3287
7      de witt          3287
8     iroquois          3287
9        piatt          3287
10       clark          3287
11       coles          3287
12  cumberland          3287
13        ford          3287
14      mclean          3287
15   vermilion          3287


In [12]:
print(con.execute(f"""
SELECT MIN(AQI), MAX(AQI)
FROM '{AQI_FILE}'
""").df())

   min(AQI)  max(AQI)
0         0       289


In [13]:
AQI_FILE = '/projects/illinois/ovcri/ncsa/cs2018/idph-ch-datasets/climate/interpolated_aqi_il_2017-24/aqi_all_years.parquet'

con.execute(f"""
COPY (
    SELECT *
    FROM '{AQI_FILE}'
    WHERE LOWER(REPLACE(TRIM(region), ' ', '')) IN (
        'champaign','clark','coles','cumberland','dewitt','douglas','edgar',
        'ford','iroquois','livingston','piatt','macon','mclean',
        'moultrie','shelby','vermilion'
    )
    AND YEAR(climate_date) BETWEEN 2017 AND 2025
) TO 'aqi_filtered.parquet' (FORMAT PARQUET)
""")

In [14]:
con.execute("""
CREATE OR REPLACE TABLE aqi_clean AS
SELECT 
    LOWER(REPLACE(TRIM(region), ' ', '')) AS county,
    climate_date AS date,
    AQI
FROM 'aqi_filtered.parquet'
""")

In [15]:
con.execute("""
CREATE OR REPLACE TABLE aqi_agg AS
SELECT 
    county,
    date,
    AVG(AQI) AS avg_aqi
FROM aqi_clean
GROUP BY county, date
""")

In [16]:
con.execute("COPY aqi_agg TO 'aqi_agg.parquet' (FORMAT PARQUET)")